# 04 · Blob storage the real way

## Goal

Get scanned supplier addenda, sitting in Azure Blob Storage, grounded in the
agent — via the two-hop path that actually exists: Storage → AI Search
indexer → attach as an Azure AI Search knowledge source. There is no native
blob source; this notebook exists because that surprises people.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from csx.config import load_settings
settings = load_settings()
settings.require("AZURE_SUBSCRIPTION_ID", "AZURE_RESOURCE_GROUP", "AZURE_LOCATION")
print("Azure settings present")


## Concept

**Finding #7 corrects a real assumption.** The supported Copilot Studio
knowledge sources are: upload, public website, SharePoint, Azure AI Search,
Dataverse, Dynamics 365, Salesforce, and ServiceNow. Blob storage is not on
that list. What blob storage *can* do is sit behind an Azure AI Search
indexer + skillset, and the resulting search index attaches as an **Azure
AI Search** knowledge source — that's the two-hop shape.

This matters for capacity planning as much as architecture: every blob
source in your estate now implies a Search service, an indexer refresh
schedule, and a skillset to run OCR/chunking on scanned PDFs (the addenda
here are scans, not text PDFs — that's deliberate, so the skillset step
isn't skippable). Refresh semantics are the indexer's, not the knowledge
source's: re-indexing on a schedule, not on every Storage write.


## Build


### Deploy Storage + AI Search (Bicep — see infra/bicep)


In [ ]:
import subprocess, json
deploy = subprocess.run([
    "az", "deployment", "group", "create",
    "--resource-group", settings.get("AZURE_RESOURCE_GROUP"),
    "--template-file", "../infra/bicep/main.bicep",
    "--parameters", "namePrefix=crd-dev", f"location={settings.get('AZURE_LOCATION')}",
], capture_output=True, text=True)
outputs = json.loads(deploy.stdout)["properties"]["outputs"] if deploy.returncode == 0 else {}
print(outputs)


### Upload the scanned addenda


In [ ]:
from azure.storage.blob import BlobServiceClient

conn_str = outputs["storageAccountName"]["value"]  # or read from Key Vault via infra/bicep/modules/keyvault.bicep
blob_service = BlobServiceClient.from_connection_string(settings.get("STORAGE_ACCOUNT_NAME"))
container = blob_service.get_container_client("supplier-addenda")

for scan in ["northwind-pricing-addendum-scan.pdf"]:
    with open(f"sample_data/{scan}", "rb") as f:
        container.upload_blob(scan, f, overwrite=True)  # overwrite=True keeps this idempotent
print("addenda uploaded")


### Indexer + skillset — OCR the scans, then index


In [ ]:
from azure.search.documents.indexes import SearchIndexerClient
from azure.search.documents.indexes.models import (
    SearchIndexerDataSourceConnection, SearchIndexerSkillset, OcrSkill, SearchIndexer,
)
from azure.core.credentials import AzureKeyCredential

indexer_client = SearchIndexerClient(settings.get("AI_SEARCH_ENDPOINT"), AzureKeyCredential(settings.get("AI_SEARCH_ADMIN_KEY")))

# get-or-create pattern throughout — idempotent re-run
def get_or_create(client, name, get_fn, create_fn, obj):
    try:
        return get_fn(name)
    except Exception:
        return create_fn(obj)

data_source = SearchIndexerDataSourceConnection(
    name="crd-addenda-blob", type="azureblob",
    connection_string=settings.get("STORAGE_ACCOUNT_NAME"),
    container={"name": "supplier-addenda"},
)
get_or_create(indexer_client, "crd-addenda-blob", indexer_client.get_data_source_connection,
              indexer_client.create_data_source_connection, data_source)

skillset = SearchIndexerSkillset(
    name="crd-addenda-ocr",
    skills=[OcrSkill(context="/document/normalized_images/*", inputs=[], outputs=[])],
)
get_or_create(indexer_client, "crd-addenda-ocr", indexer_client.get_skillset,
              indexer_client.create_skillset, skillset)

indexer = SearchIndexer(name="crd-addenda-indexer", data_source_name="crd-addenda-blob",
                         skillset_name="crd-addenda-ocr", target_index_name="crd-addenda-index",
                         schedule={"interval": "PT2H"})  # refresh every 2 hours — not on every blob write
get_or_create(indexer_client, "crd-addenda-indexer", indexer_client.get_indexer,
              indexer_client.create_indexer, indexer)

indexer_client.run_indexer("crd-addenda-indexer")
print("indexer running — refresh semantics: PT2H schedule, not event-driven")


### Attach the index as knowledge


In [ ]:
import yaml
from pathlib import Path
workspace = Path("../agents/contract-renewal-desk")

source = {
    "id": "addenda-search-index",
    "type": "azure_ai_search",
    "displayName": "Supplier addenda (scanned, OCR'd)",
    "description": "Pricing and terms addenda scanned from paper originals — Northwind Fasteners and others.",
    "endpoint": settings.get("AI_SEARCH_ENDPOINT"),
    "indexName": "crd-addenda-index",
}
(workspace / "knowledge" / "addenda-search-index.yaml").write_text(yaml.dump(source, sort_keys=False))

from csx.pac import copilot_push
import subprocess
copilot_push(workspace)
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)


## Verify

Same harness, same golden set, every notebook.


In [ ]:
from csx.clients import get_copilot_client
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter

client = get_copilot_client(settings, delegated=True)
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))
cases = load_golden(tags=["core"]) + load_golden(tags=["blob"])
suite = run_suite(client, cases=cases, credit_meter=meter, min_pass_rate=0.8)


## Cost


In [ ]:
meter.report_cost("04", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=suite.total_credits, note="AI Search indexer build + attach + eval")


## Teardown


In [ ]:
# Search/indexer resources stay — the spine agent depends on them through 25.
# If you're only exploring this notebook standalone, tear down with:
#   az deployment group delete --resource-group <rg> --name <deployment>
print("Search + Storage resources persist for the rest of the curriculum.")
